In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("auto.csv")
df = df.set_index("ID")
df.head()

,CarNumber,Make_n_model,Refund,Fines,History
ID,,,,,
0,Y163O8161RUS,Ford Focus,2.0,3200.0,NaN
1,E432XX77RUS,Toyota Camry,1.0,6500.0,NaN
2,7184TT36RUS,Ford Focus,1.0,2100.0,NaN
3,X582HE161RUS,Ford Focus,2.0,2000.0,NaN
4,E34877152RUS,Ford Focus,2.0,6100.0,NaN


In [3]:
df.count()

CarNumber       931
Make_n_model    931
Refund          914
Fines           869
History          82
dtype: int64

In [4]:
dedup_df = df.drop_duplicates(
    subset=["CarNumber", "Make_n_model", "Fines"], 
    keep="last", 
    inplace=False
)

In [5]:
dedup_df.count()

CarNumber       725
Make_n_model    725
Refund          713
Fines           665
History          65
dtype: int64

In [6]:
duplicates_removed_count = len(df) - len(dedup_df)
print(duplicates_removed_count)

206


In [7]:
df.isna().sum()

CarNumber         0
Make_n_model      0
Refund           17
Fines            62
History         849
dtype: int64

In [8]:
dedup_df.isna().sum()

CarNumber         0
Make_n_model      0
Refund           12
Fines            60
History         660
dtype: int64

In [9]:
clean_df = dedup_df.dropna(axis=1, thresh=len(dedup_df) - 500)
clean_df.isna().sum()

CarNumber        0
Make_n_model     0
Refund          12
Fines           60
dtype: int64

In [10]:
clean_df["Refund"] = clean_df["Refund"].ffill()
clean_df.isna().sum()

CarNumber        0
Make_n_model     0
Refund           0
Fines           60
dtype: int64

In [11]:
mean_value = clean_df["Fines"].mean()
mean_value

np.float64(8594.586466165414)

In [12]:
clean_df["Fines"] = clean_df["Fines"].fillna(mean_value)
clean_df.isna().sum()

CarNumber       0
Make_n_model    0
Refund          0
Fines           0
dtype: int64

In [13]:
clean_df["Make"] = clean_df["Make_n_model"].apply(
    lambda value: value.split(" ", 1)[0] if isinstance(value, str) else value
)

clean_df["Model"] = clean_df["Make_n_model"].apply(
    lambda value: (
        value.split(" ", 1)[1] if isinstance(value, str) and " " in value else ""
    )
)

clean_df.head()

,CarNumber,Make_n_model,Refund,Fines,Make,Model
ID,,,,,,
0,Y163O8161RUS,Ford Focus,2.0,3200.0,Ford,Focus
1,E432XX77RUS,Toyota Camry,1.0,6500.0,Toyota,Camry
2,7184TT36RUS,Ford Focus,1.0,2100.0,Ford,Focus
3,X582HE161RUS,Ford Focus,2.0,2000.0,Ford,Focus
5,92918M178RUS,Ford Focus,1.0,5700.0,Ford,Focus


In [14]:
clean_df = clean_df.drop("Make_n_model", axis=1)
clean_df.head()

,CarNumber,Refund,Fines,Make,Model
ID,,,,,
0,Y163O8161RUS,2.0,3200.0,Ford,Focus
1,E432XX77RUS,1.0,6500.0,Toyota,Camry
2,7184TT36RUS,1.0,2100.0,Ford,Focus
3,X582HE161RUS,2.0,2000.0,Ford,Focus
5,92918M178RUS,1.0,5700.0,Ford,Focus


In [15]:
clean_df.to_json("auto.json", orient="records", indent=4)